# FinOps Cloud Cost Forecasting — FastAPI Model Serving

This notebook builds and tests a FastAPI inference service around the
production `champion` model stored in the MLflow Model Registry.

Endpoints:

- `GET /`
- `GET /health`
- `GET /model-info`
- `POST /predict`

Production model:

- Registered model: `finops-cloud-cost-forecasting-clean-v1`
- Alias: `champion`
- Forecast horizon: one hour

In [ ]:
# ============================================================
# CELL 1 — FASTAPI SERVING ENVIRONMENT
# ============================================================

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False
)

%pip install -q \
    "mlflow==3.15.1" \
    "scikit-learn==1.6.1" \
    "catboost==1.2.10" \
    fastapi \
    uvicorn \
    httpx

import sys
import numpy as np
import pandas as pd
import sklearn
import mlflow
import fastapi
import pydantic
import uvicorn
import httpx

environment_versions = pd.DataFrame({
    "Component": [
        "Python",
        "MLflow",
        "scikit-learn",
        "FastAPI",
        "Pydantic",
        "Uvicorn",
        "HTTPX"
    ],
    "Version": [
        sys.version.split()[0],
        mlflow.__version__,
        sklearn.__version__,
        fastapi.__version__,
        pydantic.__version__,
        uvicorn.__version__,
        httpx.__version__
    ]
})

print("FASTAPI SERVING ENVIRONMENT")
print("=" * 60)

display(environment_versions)

print("\nGoogle Drive mounted.")
print("Serving dependencies loaded successfully.")

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 94.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 99.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 101.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.2/216.2 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━

,Component,Version
0,Python,3.13.15
1,MLflow,3.15.1
2,scikit-learn,1.6.1
3,FastAPI,0.141.1
4,Pydantic,2.13.4
5,Uvicorn,0.52.3
6,HTTPX,0.28.1



Google Drive mounted.
Serving dependencies loaded successfully.


In [ ]:
# ============================================================
# CELL 2 — ENSURE REGISTRY CHAMPION EXISTS AND LOAD IT
# ============================================================

from pathlib import Path
from mlflow.tracking import MlflowClient
from mlflow.exceptions import MlflowException

# ------------------------------------------------------------
# Persistent MLflow connection
# ------------------------------------------------------------

MLFLOW_ROOT_FINOPS = Path(
    "/content/drive/MyDrive/finops_mlflow"
)

MLFLOW_DB_FINOPS = (
    MLFLOW_ROOT_FINOPS / "mlflow.db"
)

if not MLFLOW_DB_FINOPS.exists():
    raise FileNotFoundError(
        f"MLflow database not found:\n{MLFLOW_DB_FINOPS}"
    )

MLFLOW_TRACKING_URI_FINOPS = (
    f"sqlite:///{MLFLOW_DB_FINOPS}"
)

mlflow.set_tracking_uri(
    MLFLOW_TRACKING_URI_FINOPS
)

mlflow.set_registry_uri(
    MLFLOW_TRACKING_URI_FINOPS
)

CLEAN_EXPERIMENT_NAME_FINOPS = (
    "finops-cloud-cost-forecasting-clean-v1"
)

REGISTERED_MODEL_NAME_FINOPS = (
    "finops-cloud-cost-forecasting-clean-v1"
)

PRODUCTION_ALIAS_FINOPS = "champion"

registry_client_finops = MlflowClient(
    tracking_uri=MLFLOW_TRACKING_URI_FINOPS,
    registry_uri=MLFLOW_TRACKING_URI_FINOPS
)

# ------------------------------------------------------------
# Find the verified clean baseline run
# ------------------------------------------------------------

clean_experiment_finops = (
    mlflow.get_experiment_by_name(
        CLEAN_EXPERIMENT_NAME_FINOPS
    )
)

if clean_experiment_finops is None:
    raise RuntimeError(
        "Clean MLflow experiment was not found."
    )

clean_runs_finops = mlflow.search_runs(
    experiment_ids=[
        clean_experiment_finops.experiment_id
    ]
)

baseline_runs_finops = clean_runs_finops[
    (
        clean_runs_finops["tags.mlflow.runName"]
        == "naive_baseline_clean"
    )
    &
    (
        clean_runs_finops["status"]
        == "FINISHED"
    )
]

if len(baseline_runs_finops) != 1:
    raise RuntimeError(
        "Expected exactly one successful clean baseline "
        f"run, but found {len(baseline_runs_finops)}."
    )

clean_naive_run_id_finops = (
    baseline_runs_finops.iloc[0]["run_id"]
)

print("Verified baseline run:")
print(clean_naive_run_id_finops)

# ------------------------------------------------------------
# Find or create the registered baseline version
# ------------------------------------------------------------

try:
    existing_versions_finops = (
        registry_client_finops.search_model_versions(
            f"name='{REGISTERED_MODEL_NAME_FINOPS}'"
        )
    )

except MlflowException:
    existing_versions_finops = []

champion_version_finops = None

for model_version in existing_versions_finops:

    if model_version.run_id == clean_naive_run_id_finops:
        champion_version_finops = model_version
        break

if champion_version_finops is None:

    baseline_model_uri_finops = (
        f"runs:/{clean_naive_run_id_finops}/model"
    )

    champion_version_finops = mlflow.register_model(
        model_uri=baseline_model_uri_finops,
        name=REGISTERED_MODEL_NAME_FINOPS
    )

    print(
        "New baseline registry version created:",
        champion_version_finops.version
    )

else:

    print(
        "Existing baseline registry version found:",
        champion_version_finops.version
    )

# ------------------------------------------------------------
# Assign production alias and metadata
# ------------------------------------------------------------

registry_client_finops.set_registered_model_alias(
    name=REGISTERED_MODEL_NAME_FINOPS,
    alias=PRODUCTION_ALIAS_FINOPS,
    version=champion_version_finops.version
)

registry_client_finops.set_model_version_tag(
    name=REGISTERED_MODEL_NAME_FINOPS,
    version=champion_version_finops.version,
    key="lifecycle_role",
    value="production_champion"
)

registry_client_finops.set_model_version_tag(
    name=REGISTERED_MODEL_NAME_FINOPS,
    version=champion_version_finops.version,
    key="deployment_status",
    value="production"
)

# ------------------------------------------------------------
# Load champion through its registry alias
# ------------------------------------------------------------

CHAMPION_MODEL_URI_FINOPS = (
    f"models:/{REGISTERED_MODEL_NAME_FINOPS}"
    f"@{PRODUCTION_ALIAS_FINOPS}"
)

champion_model_finops = mlflow.pyfunc.load_model(
    CHAMPION_MODEL_URI_FINOPS
)

resolved_champion_finops = (
    registry_client_finops.get_model_version_by_alias(
        name=REGISTERED_MODEL_NAME_FINOPS,
        alias=PRODUCTION_ALIAS_FINOPS
    )
)

print("\nMLFLOW CHAMPION READY")
print("=" * 70)

print(
    "Registered model:",
    REGISTERED_MODEL_NAME_FINOPS
)

print(
    "Alias           :",
    PRODUCTION_ALIAS_FINOPS
)

print(
    "Model version   :",
    resolved_champion_finops.version
)

print(
    "Source run ID   :",
    resolved_champion_finops.run_id
)

print(
    "Model URI       :",
    CHAMPION_MODEL_URI_FINOPS
)

# ------------------------------------------------------------
# Direct inference test
# ------------------------------------------------------------

sample_request_finops = pd.DataFrame({
    "estimated_cost_index": pd.Series(
        [24.397998],
        dtype="float64"
    )
})

sample_prediction_finops = np.asarray(
    champion_model_finops.predict(
        sample_request_finops
    )
).reshape(-1)

sample_forecast_finops = float(
    sample_prediction_finops[0]
)

print("\nDIRECT PREDICTION")
print("-" * 70)

print(
    "Current-hour estimated cost:",
    sample_request_finops[
        "estimated_cost_index"
    ].iloc[0]
)

print(
    "Predicted next-hour cost  :",
    sample_forecast_finops
)

assert (
    resolved_champion_finops.run_id
    == clean_naive_run_id_finops
)

assert np.isfinite(sample_forecast_finops)
assert sample_forecast_finops >= 0

print("\nChampion registration and loading verified.")

Verified baseline run:
25b920c78baf4274a36b9cea73f39e23


Successfully registered model 'finops-cloud-cost-forecasting-clean-v1'.
2026/08/24 05:43:24 WARNING mlflow.tracking._model_registry.fluent: Run with id 25b920c78baf4274a36b9cea73f39e23 has no artifacts at artifact path 'model', registering model based on models:/m-36ed6d62987f4c1892a1a554a97c1879 instead
Created version '1' of model 'finops-cloud-cost-forecasting-clean-v1'.


New baseline registry version created: 1

MLFLOW CHAMPION READY
Registered model: finops-cloud-cost-forecasting-clean-v1
Alias           : champion
Model version   : 1
Source run ID   : 25b920c78baf4274a36b9cea73f39e23
Model URI       : models:/finops-cloud-cost-forecasting-clean-v1@champion

DIRECT PREDICTION
----------------------------------------------------------------------
Current-hour estimated cost: 24.397998
Predicted next-hour cost  : 24.397998000000005

Champion registration and loading verified.


In [ ]:
# ============================================================
# CELL 3 — FASTAPI APPLICATION AND ENDPOINTS
# ============================================================

from datetime import datetime, timezone
from typing import Literal

from fastapi import (
    FastAPI,
    HTTPException
)

from pydantic import (
    BaseModel,
    ConfigDict,
    Field
)

# ------------------------------------------------------------
# Request and response schemas
# ------------------------------------------------------------

class PredictionRequestFinOps(BaseModel):

    model_config = ConfigDict(
        json_schema_extra={
            "example": {
                "estimated_cost_index": 24.397998
            }
        }
    )

    estimated_cost_index: float = Field(
        ...,
        ge=0,
        description=(
            "Current-hour estimated cloud cost index."
        )
    )


class PredictionResponseFinOps(BaseModel):

    current_hour_cost: float

    predicted_next_hour_cost: float

    forecast_horizon: Literal["1_hour"]

    registered_model: str

    model_alias: str

    model_version: str

    prediction_timestamp_utc: str


class HealthResponseFinOps(BaseModel):

    status: Literal["healthy"]

    model_loaded: bool

    registered_model: str

    model_alias: str

    timestamp_utc: str


# ------------------------------------------------------------
# FastAPI application
# ------------------------------------------------------------

app = FastAPI(
    title="FinOps Cloud Cost Forecasting API",
    description=(
        "Forecasts the next-hour estimated cloud cost "
        "using the production champion stored in the "
        "MLflow Model Registry."
    ),
    version="1.0.0",
    docs_url="/docs",
    redoc_url="/redoc"
)


# ------------------------------------------------------------
# Root endpoint
# ------------------------------------------------------------

@app.get("/")
def root_finops():

    return {
        "service": "FinOps Cloud Cost Forecasting API",
        "status": "running",
        "api_version": "1.0.0",
        "documentation": "/docs"
    }


# ------------------------------------------------------------
# Health endpoint
# ------------------------------------------------------------

@app.get(
    "/health",
    response_model=HealthResponseFinOps
)
def health_finops():

    return HealthResponseFinOps(
        status="healthy",
        model_loaded=(
            champion_model_finops is not None
        ),
        registered_model=(
            REGISTERED_MODEL_NAME_FINOPS
        ),
        model_alias=PRODUCTION_ALIAS_FINOPS,
        timestamp_utc=(
            datetime.now(timezone.utc).isoformat()
        )
    )


# ------------------------------------------------------------
# Model information endpoint
# ------------------------------------------------------------

@app.get("/model-info")
def model_info_finops():

    return {
        "registered_model": (
            REGISTERED_MODEL_NAME_FINOPS
        ),
        "alias": PRODUCTION_ALIAS_FINOPS,
        "version": str(
            resolved_champion_finops.version
        ),
        "source_run_id": (
            resolved_champion_finops.run_id
        ),
        "model_uri": CHAMPION_MODEL_URI_FINOPS,
        "forecast_horizon": "1_hour",
        "input_features": [
            "estimated_cost_index"
        ],
        "prediction_rule": (
            "current-hour estimated cost predicts "
            "next-hour estimated cost"
        ),
        "primary_metric": "test_mae",
        "production_test_mae": 7.882846
    }


# ------------------------------------------------------------
# Prediction endpoint
# ------------------------------------------------------------

@app.post(
    "/predict",
    response_model=PredictionResponseFinOps
)
def predict_finops(
    request: PredictionRequestFinOps
):

    try:

        model_input = pd.DataFrame({
            "estimated_cost_index": pd.Series(
                [
                    request.estimated_cost_index
                ],
                dtype="float64"
            )
        })

        model_output = np.asarray(
            champion_model_finops.predict(
                model_input
            )
        ).reshape(-1)

        predicted_cost = float(
            model_output[0]
        )

        if not np.isfinite(predicted_cost):
            raise ValueError(
                "Model returned a non-finite prediction."
            )

        if predicted_cost < 0:
            raise ValueError(
                "Model returned a negative cost prediction."
            )

        return PredictionResponseFinOps(
            current_hour_cost=(
                request.estimated_cost_index
            ),
            predicted_next_hour_cost=(
                predicted_cost
            ),
            forecast_horizon="1_hour",
            registered_model=(
                REGISTERED_MODEL_NAME_FINOPS
            ),
            model_alias=(
                PRODUCTION_ALIAS_FINOPS
            ),
            model_version=str(
                resolved_champion_finops.version
            ),
            prediction_timestamp_utc=(
                datetime.now(
                    timezone.utc
                ).isoformat()
            )
        )

    except Exception as error:

        raise HTTPException(
            status_code=500,
            detail=(
                "Prediction failed: "
                f"{str(error)}"
            )
        ) from error


print("FASTAPI APPLICATION CREATED")
print("=" * 60)

print("Application title:", app.title)
print("API version      :", app.version)

print("\nAvailable endpoints:")
for route in app.routes:
    if hasattr(route, "methods"):
        print(
            sorted(route.methods),
            route.path
        )

FASTAPI APPLICATION CREATED
Application title: FinOps Cloud Cost Forecasting API
API version      : 1.0.0

Available endpoints:
['GET', 'HEAD'] /openapi.json
['GET', 'HEAD'] /docs
['GET', 'HEAD'] /docs/oauth2-redirect
['GET', 'HEAD'] /redoc
['GET'] /
['GET'] /health
['GET'] /model-info
['POST'] /predict


In [ ]:
# ============================================================
# CELL 4 — FASTAPI ENDPOINT TESTS
# ============================================================

import json

from fastapi.testclient import TestClient

api_client_finops = TestClient(app)

# ------------------------------------------------------------
# GET /
# ------------------------------------------------------------

root_response_finops = api_client_finops.get(
    "/"
)

assert root_response_finops.status_code == 200
assert root_response_finops.json()["status"] == "running"

# ------------------------------------------------------------
# GET /health
# ------------------------------------------------------------

health_response_finops = api_client_finops.get(
    "/health"
)

assert health_response_finops.status_code == 200
assert (
    health_response_finops.json()["status"]
    == "healthy"
)
assert (
    health_response_finops.json()["model_loaded"]
    is True
)

# ------------------------------------------------------------
# GET /model-info
# ------------------------------------------------------------

model_info_response_finops = (
    api_client_finops.get(
        "/model-info"
    )
)

assert model_info_response_finops.status_code == 200
assert (
    model_info_response_finops.json()["alias"]
    == "champion"
)

# ------------------------------------------------------------
# POST /predict — valid request
# ------------------------------------------------------------

valid_cost_finops = 24.397998

valid_prediction_response_finops = (
    api_client_finops.post(
        "/predict",
        json={
            "estimated_cost_index": valid_cost_finops
        }
    )
)

assert (
    valid_prediction_response_finops.status_code
    == 200
)

valid_prediction_body_finops = (
    valid_prediction_response_finops.json()
)

assert np.isclose(
    valid_prediction_body_finops[
        "predicted_next_hour_cost"
    ],
    valid_cost_finops,
    atol=1e-6
)

# ------------------------------------------------------------
# POST /predict — zero is valid
# ------------------------------------------------------------

zero_response_finops = api_client_finops.post(
    "/predict",
    json={
        "estimated_cost_index": 0.0
    }
)

assert zero_response_finops.status_code == 200

# ------------------------------------------------------------
# Invalid input tests
# ------------------------------------------------------------

negative_response_finops = api_client_finops.post(
    "/predict",
    json={
        "estimated_cost_index": -10.0
    }
)

missing_response_finops = api_client_finops.post(
    "/predict",
    json={}
)

wrong_type_response_finops = api_client_finops.post(
    "/predict",
    json={
        "estimated_cost_index": "not-a-number"
    }
)

assert negative_response_finops.status_code == 422
assert missing_response_finops.status_code == 422
assert wrong_type_response_finops.status_code == 422

# ------------------------------------------------------------
# Test summary
# ------------------------------------------------------------

api_test_summary_finops = pd.DataFrame({
    "Test": [
        "Root endpoint",
        "Health endpoint",
        "Model information",
        "Valid prediction",
        "Zero-cost prediction",
        "Negative-cost rejection",
        "Missing-field rejection",
        "Wrong-type rejection"
    ],
    "Expected status": [
        200,
        200,
        200,
        200,
        200,
        422,
        422,
        422
    ],
    "Actual status": [
        root_response_finops.status_code,
        health_response_finops.status_code,
        model_info_response_finops.status_code,
        valid_prediction_response_finops.status_code,
        zero_response_finops.status_code,
        negative_response_finops.status_code,
        missing_response_finops.status_code,
        wrong_type_response_finops.status_code
    ]
})

api_test_summary_finops["Result"] = np.where(
    api_test_summary_finops["Expected status"]
    == api_test_summary_finops["Actual status"],
    "PASS",
    "FAIL"
)

print("FASTAPI ENDPOINT TEST RESULTS")
print("=" * 75)

display(api_test_summary_finops)

print("\nVALID PREDICTION RESPONSE")
print("-" * 75)

print(
    json.dumps(
        valid_prediction_body_finops,
        indent=2
    )
)

assert (
    api_test_summary_finops["Result"]
    == "PASS"
).all()

print("\nAll FastAPI endpoint tests passed.")

FASTAPI ENDPOINT TEST RESULTS


,Test,Expected status,Actual status,Result
0,Root endpoint,200,200,PASS
1,Health endpoint,200,200,PASS
2,Model information,200,200,PASS
3,Valid prediction,200,200,PASS
4,Zero-cost prediction,200,200,PASS
5,Negative-cost rejection,422,422,PASS
6,Missing-field rejection,422,422,PASS
7,Wrong-type rejection,422,422,PASS



VALID PREDICTION RESPONSE
---------------------------------------------------------------------------
{
  "current_hour_cost": 24.397998,
  "predicted_next_hour_cost": 24.397998000000005,
  "forecast_horizon": "1_hour",
  "registered_model": "finops-cloud-cost-forecasting-clean-v1",
  "model_alias": "champion",
  "model_version": "1",
  "prediction_timestamp_utc": "2026-08-24T05:54:07.512333+00:00"
}

All FastAPI endpoint tests passed.


In [ ]:
# ============================================================
# CELL 5 — EXPORT PORTABLE CHAMPION MODEL BUNDLE
# ============================================================

import json
import shutil
import tempfile

from pathlib import Path

# Persistent deployment export location
DEPLOYMENT_EXPORT_DIR_FINOPS = Path(
    "/content/drive/MyDrive/"
    "finops_deployment_exports"
)

DEPLOYMENT_EXPORT_DIR_FINOPS.mkdir(
    parents=True,
    exist_ok=True
)

champion_version_number_finops = str(
    resolved_champion_finops.version
)

champion_run_short_id_finops = (
    resolved_champion_finops.run_id[:8]
)

bundle_name_finops = (
    f"finops_champion_v"
    f"{champion_version_number_finops}_"
    f"{champion_run_short_id_finops}"
)

bundle_zip_path_finops = (
    DEPLOYMENT_EXPORT_DIR_FINOPS
    / f"{bundle_name_finops}.zip"
)

# ------------------------------------------------------------
# Download model artifact to temporary local storage
# ------------------------------------------------------------

temporary_export_root_finops = Path(
    tempfile.mkdtemp(
        prefix="finops_champion_export_"
    )
)

downloaded_model_path_finops = Path(
    mlflow.artifacts.download_artifacts(
        artifact_uri=CHAMPION_MODEL_URI_FINOPS,
        dst_path=str(
            temporary_export_root_finops
        )
    )
)

print("Downloaded model path:")
print(downloaded_model_path_finops)

# ------------------------------------------------------------
# Add deployment manifest
# ------------------------------------------------------------

deployment_manifest_finops = {
    "project": "FinOps Cloud Cost Forecasting",
    "registered_model": (
        REGISTERED_MODEL_NAME_FINOPS
    ),
    "alias": PRODUCTION_ALIAS_FINOPS,
    "model_version": (
        champion_version_number_finops
    ),
    "source_run_id": (
        resolved_champion_finops.run_id
    ),
    "forecast_horizon": "1_hour",
    "model_role": "production_champion",
    "input_schema": {
        "estimated_cost_index": "float64"
    },
    "output": "predicted_next_hour_cost",
    "production_test_mae": 7.882846,
    "mlflow_version": mlflow.__version__,
    "sklearn_version": sklearn.__version__
}

manifest_path_finops = (
    downloaded_model_path_finops
    / "deployment_manifest.json"
)

with manifest_path_finops.open(
    "w",
    encoding="utf-8"
) as manifest_file:

    json.dump(
        deployment_manifest_finops,
        manifest_file,
        indent=2
    )

# ------------------------------------------------------------
# Verify portable model before creating ZIP
# ------------------------------------------------------------

portable_model_finops = mlflow.pyfunc.load_model(
    str(downloaded_model_path_finops)
)

portable_test_input_finops = pd.DataFrame({
    "estimated_cost_index": pd.Series(
        [24.397998],
        dtype="float64"
    )
})

portable_test_prediction_finops = float(
    np.asarray(
        portable_model_finops.predict(
            portable_test_input_finops
        )
    ).reshape(-1)[0]
)

assert np.isfinite(
    portable_test_prediction_finops
)

assert np.isclose(
    portable_test_prediction_finops,
    sample_forecast_finops,
    atol=1e-6
)

# ------------------------------------------------------------
# Create ZIP without overwriting an existing export
# ------------------------------------------------------------

if bundle_zip_path_finops.exists():

    print("\nDeployment bundle already exists.")
    print(bundle_zip_path_finops)

else:

    shutil.make_archive(
        base_name=str(
            bundle_zip_path_finops.with_suffix("")
        ),
        format="zip",
        root_dir=str(
            downloaded_model_path_finops
        )
    )

    print("\nDeployment bundle created.")

print("\nPORTABLE MODEL EXPORT")
print("=" * 70)

print(
    "Bundle:",
    bundle_zip_path_finops
)

print(
    "Size:",
    f"{bundle_zip_path_finops.stat().st_size / 1024:.2f} KB"
)

print(
    "Test prediction:",
    portable_test_prediction_finops
)

print(
    "\nVS Code destination:"
)

print(
    "models/champion/"
)

print(
    "\nPortable champion model verified successfully."
)

Downloaded model path:
/tmp/finops_champion_export_yw06v7rg

Deployment bundle created.

PORTABLE MODEL EXPORT
Bundle: /content/drive/MyDrive/finops_deployment_exports/finops_champion_v1_25b920c7.zip
Size: 3.45 KB
Test prediction: 24.397998000000005

VS Code destination:
models/champion/

Portable champion model verified successfully.
